# 14 Fastapi Rag Api

**Project:** Enterprise Multi-Document RAG Assistant

**Notebook:** `14-fastapi-rag-api.ipynb`

In [1]:
# Start coding here
# =====================================================
# Notebook 14
# FastAPI Enterprise RAG API
# =====================================================

from fastapi import FastAPI
from pydantic import BaseModel

import pandas as pd
import numpy as np
import pickle
import faiss

from rank_bm25 import BM25Okapi

from transformers import pipeline

from sentence_transformers import SentenceTransformer, CrossEncoder

In [2]:
app = FastAPI(
    title="Enterprise RAG API",
    version="1.0.0",
    description="Multi Document Conversational Assistant",
)

In [3]:
class QuestionRequest(BaseModel):

    question: str

In [4]:
class Citation(BaseModel):

    document: str

    department: str

In [5]:
class AnswerResponse(BaseModel):

    answer: str

    citations: list

In [6]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

llm = pipeline("text2text-generation", model="google/flan-t5-base")

Device set to use cpu


In [7]:
children_df = pd.read_csv("artifacts/child_chunks.csv")

parents_df = pd.read_csv("artifacts/parent_chunks.csv")

metadata_df = pd.read_csv("artifacts/document_metadata.csv")

In [8]:
with open("artifacts/router_embeddings.pkl", "rb") as f:

    department_embeddings = pickle.load(f)

In [9]:
vector_indexes = {}

for dept in ["hr", "finance", "engineering"]:

    vector_indexes[dept] = faiss.read_index(f"artifacts/faiss/{dept}.index")

In [10]:
with open("artifacts/faiss/chunk_mapping.pkl", "rb") as f:

    chunk_mappings = pickle.load(f)

In [11]:
children_df = children_df.merge(
    metadata_df[["document_id", "department"]], on="document_id", how="left"
)

department_bm25 = {}

department_docs = {}

In [12]:
for dept in children_df["department"].unique():

    subset = children_df[children_df["department"] == dept]

    corpus = [text.lower().split() for text in subset["content"]]

    department_bm25[dept] = BM25Okapi(corpus)

    department_docs[dept] = subset

In [13]:
conversation_memory = []

In [14]:
def rewrite_query(question):

    history = "\n".join([x["content"] for x in conversation_memory[-6:]])

    prompt = f"""
Rewrite into standalone question.

History:
{history}

Question:
{question}
"""

    response = llm(prompt, max_new_tokens=64)

    return response[0]["generated_text"]

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
def route_query(query):

    query_embedding = embedding_model.encode(query)

    scores = {}

    for dept, emb in department_embeddings.items():

        similarity = cosine_similarity(
            query_embedding.reshape(1, -1), emb.reshape(1, -1)
        )[0][0]

        scores[dept] = similarity

    return max(scores, key=scores.get)

In [17]:
def bm25_search(query, department, top_k=20):

    bm25 = department_bm25[department]

    docs = department_docs[department].copy()

    scores = bm25.get_scores(query.lower().split())

    docs["score"] = scores

    docs = docs.sort_values("score", ascending=False)

    return docs.head(top_k)

In [18]:
def semantic_search(query, department, top_k=20):

    query_embedding = embedding_model.encode([query]).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, ids = vector_indexes[department].search(query_embedding, top_k)

    mapping = chunk_mappings[department]

    results = []

    for score, idx in zip(scores[0], ids[0]):

        row = mapping.iloc[idx]

        results.append(
            {
                "parent_id": row["parent_id"],
                "content": row["content"],
                "score": float(score),
            }
        )

    return results

In [19]:
def hybrid_candidates(query):

    dept = route_query(query)

    bm25_docs = bm25_search(query, dept)

    semantic_docs = semantic_search(query, dept)

    candidates = {}

In [ ]:
# for _, row in bm25_docs.iterrows():

#     candidates[
#         row["parent_id"]
#     ] = row["content"]

# for row in semantic_docs:

#     candidates[
#         row["parent_id"]
#     ] = row["content"]

# return list(
#     candidates.items()
# )

NameError: name 'bm25_docs' is not defined

In [22]:
def rerank_documents(query, candidates, top_k=5):

    pairs = [[query, text] for _, text in candidates]

    scores = cross_encoder.predict(pairs)

    ranked = []

    for (doc_id, text), score in zip(candidates, scores):

        ranked.append({"parent_id": doc_id, "content": text, "score": float(score)})

    ranked = sorted(ranked, key=lambda x: x["score"], reverse=True)

    return ranked[:top_k]

In [23]:
def build_context(ranked_docs):

    context = ""

    for doc in ranked_docs:

        context += doc["content"]

        context += "\n\n"

    return context

In [24]:
def generate_answer(question, context):

    prompt = f"""
Answer only from context.

Context:

{context}

Question:

{question}

Answer:
"""

    response = llm(prompt, max_new_tokens=256)

    return response[0]["generated_text"]

In [25]:
def rag_pipeline(question):

    standalone_question = rewrite_query(question)

    candidates = hybrid_candidates(standalone_question)

    ranked_docs = rerank_documents(standalone_question, candidates)

    context = build_context(ranked_docs)

    answer = generate_answer(standalone_question, context)

    conversation_memory.append({"role": "user", "content": question})

    conversation_memory.append({"role": "assistant", "content": answer})

    return {"answer": answer, "sources": len(ranked_docs)}

In [26]:
@app.post("/ask")
def ask_question(request: QuestionRequest):

    result = rag_pipeline(request.question)

    return result

In [27]:
@app.get("/health")
def health():

    return {"status": "healthy"}

In [29]:
!pip install nest_asyncio


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# import nest_asyncio

# nest_asyncio.apply()

# import uvicorn

# uvicorn.run(app, host="0.0.0.0", port=8000)

RuntimeError: Cannot run the event loop while another loop is running

In [28]:
if __name__ == "__main__":

    import uvicorn

    uvicorn.run(app, host="0.0.0.0", port=8000)

RuntimeError: Cannot run the event loop while another loop is running